In [4]:
import json
from IPython.display import JSON

# 1. Path to your file
file_path = 'data/responses.jsonl'

# 2. Load the lines into a list of dictionaries
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():  # Skip empty lines
            data.append(json.loads(line))

# 3. View the first record (or the whole list) interactively
# We wrap it in a key like 'root' to make it easier to navigate
display(data)

[{'question_id': 5,
  'question': 'How is the chair of Food Standards Scotland appointed?',
  'llm_name': 'mistral-large-3:675b-cloud',
  'timestamp': '2026-02-17T11:20:45.762077',
  'deep_research': False,
  'test_case': {'input': 'How is the chair of Food Standards Scotland appointed?',
   'actual_output': 'Here is the statutory process for appointing the **chair of Food Standards Scotland** under the **Food (Scotland) Act 2015**:\n\n---\n\n### **1. Appointment Authority**\n- The **chair is appointed by the Scottish Ministers**.\n  - **Source:** [Food (Scotland) Act 2015 - s.6(1)(a)](http://www.legislation.gov.uk/asp/2015/1/section/6)\n\n---\n\n### **2. Eligibility & Disqualification**\n#### **Disqualifying Factors**\nA person **cannot** be appointed as chair if they are:\n- A **member of the Scottish Parliament (MSP)**,\n- A **member of the House of Commons**,\n- A **member of the European Parliament**,\n- An **office-holder in the Scottish Administration**,\n- A **councillor of any

In [3]:
from utils.lexchat_client import get_authenticated_client

client = get_authenticated_client()

chat_payload = {
    "messages": [{"role": "user", "content": "how is the chair of food standards scotland appointed?"}],
    "model": "kimi-k2.5:cloud",
    "deep_research": False,
    "stream": True
}

with client.stream("POST", "/api/system/chat", json=chat_payload) as response:
    for line in response.iter_lines():
        print(line)

data: {"type": "tool_call", "tool_calls": [{"id": "functions.delegate_research:0", "function": {"index": 0, "name": "delegate_research", "arguments": {"query": "How is the Chair of Food Standards Scotland appointed? What is the statutory process for their appointment, including who makes the appointment and any relevant legislative provisions (particularly the Food (Scotland) Act 2015 or other applicable legislation)?"}}}]}

data: {"type": "tool_start", "tool": "Research Agent"}

data: {"type": "tool_start", "tool": "Worker: search_legislation"}

data: {"type": "api_call_start", "id": "dd86e6e1-73af-42c6-b6a8-fda250edf921", "url": "https://lex.lab.i.ai.gov.uk/legislation/search", "method": "POST", "payload": {"query": "Food (Scotland) Act 2015", "year_from": null, "year_to": null, "limit": 5, "include_text": false}}

data: {"type": "api_call_end", "id": "dd86e6e1-73af-42c6-b6a8-fda250edf921", "url": "https://lex.lab.i.ai.gov.uk/legislation/search", "status": 200, "response": {"results"

KeyboardInterrupt: 

In [9]:
client

In [ ]:
# Inspect specific events by type
import pandas as pd

# Get all tool_start events
tool_starts = [e for e in events if e.get("type") == "tool_start"]
print(f"Found {len(tool_starts)} tool_start events\n")

for i, event in enumerate(tool_starts, 1):
    print(f"\n=== Tool Start {i} ===")
    print(f"Tool field type: {type(event.get('tool'))}")
    print(f"Tool field value: {event.get('tool')}")
    print(f"Full event keys: {list(event.keys())}")
    print()

In [ ]:
# Inspect tool_end events to see the result structure
tool_ends = [e for e in events if e.get("type") == "tool_end"]
print(f"Found {len(tool_ends)} tool_end events\n")

for i, event in enumerate(tool_ends, 1):
    print(f"\n=== Tool End {i} ===")
    print(f"Tool field: {event.get('tool')}")
    print(f"Result field type: {type(event.get('result'))}")
    
    # Show first 300 chars of result
    result = event.get('result', '')
    if isinstance(result, dict):
        print(f"Result (dict keys): {list(result.keys())}")
    else:
        print(f"Result (first 300 chars): {str(result)[:300]}...")
    print()

In [ ]:
# View the complete events list as JSON for detailed inspection
from IPython.display import JSON

# Filter to just tool events
tool_events = [e for e in events if e.get("type") in ["tool_start", "tool_end", "api_call_start", "api_call_end"]]
JSON(tool_events)